In [1]:
import pandas as pd
import great_expectations as gx
import json
from datetime import datetime

In [2]:
print(gx.__version__)

0.18.21


In [3]:
df = pd.read_csv('dataset1.csv')
print(f'Строк и столбцов: {df.shape}')

Строк и столбцов: (114000, 21)


In [4]:
# удалим ненужные колонки
columns_to_drop = [col for col in df.columns if col.startswith('Unnamed') or col == 'index']
if columns_to_drop:
    df = df.drop(columns=columns_to_drop)
print(f'Строк и столбцов после очистки: {df.shape}')

Строк и столбцов после очистки: (114000, 20)


In [5]:
UNIQUE_GENRES = set(df['track_genre'].dropna().unique())
n_genres = len(UNIQUE_GENRES)
print(f'Уникальных жанров: {n_genres}')

Уникальных жанров: 114


In [6]:
import shutil
import os

context_dir = 'great_expectations'
if os.path.exists(context_dir):
    shutil.rmtree(context_dir)

context = gx.get_context(mode='file', project_root_dir=context_dir)
datasource = context.data_sources.add_pandas(name='music_data_source')
asset = datasource.add_dataframe_asset(name='music_data_asset')
batch_request = asset.build_batch_request(options={'dataframe': df})

TypeError: 'project_root_dir' and 'context_root_dir' are conflicting args; please only provide one

In [7]:
# создание ExpectationSuite
from great_expectations.core import ExpectationSuite
expectation_suite = ExpectationSuite(name='music_data_expectations')
expectation_suite = context.suites.add(expectation_suite)
print(f'Suite создан: {expectation_suite.name}')

AttributeError: module 'great_expectations' has no attribute 'ExpectationSuite'

In [7]:
# функции для добавления ожиданий
def add_type_and_length(suite, column, type_name, length):
    suite.add_expectation(gx.expectations.ExpectColumnValuesToBeOfType(column=column, type=type_name))
    suite.add_expectation(gx.expectations.ExpectColumnValueLengthsToEqual(column=column, value=length))

def add_type_and_length_range(suite, column, type_name, min_len, max_len):
    suite.add_expectation(gx.expectations.ExpectColumnValuesToBeOfType(column=column, type=type_name))
    suite.add_expectation(gx.expectations.ExpectColumnValueLengthsToBeBetween(
        column=column, min_value=min_len, max_value=max_len, strict_min=True, strict_max=True
    ))

def add_type_and_range(suite, column, type_name, min_val, max_val, strict_min=False, strict_max=False):
    suite.add_expectation(gx.expectations.ExpectColumnValuesToBeOfType(column=column, type=type_name))
    suite.add_expectation(gx.expectations.ExpectColumnValuesToBeBetween(
        column=column, min_value=min_val, max_value=max_val, strict_min=strict_min, strict_max=strict_max
    ))

In [8]:
# Проверка обязательных колонок
required_columns = [
    'track_id', 'artists', 'album_name', 'track_name', 'popularity',
    'duration_ms', 'explicit', 'danceability', 'energy', 'key',
    'loudness', 'mode', 'speechiness', 'acousticness', 'instrumentalness',
    'liveness', 'valence', 'tempo', 'time_signature', 'track_genre'
]

for col in required_columns:
    expectation_suite.add_expectation(
        gx.expectations.ExpectColumnToExist(column=col)
    )

In [9]:
# Проверка на отсутствие пропусков
for col in required_columns:
    expectation_suite.add_expectation(
        gx.expectations.ExpectColumnValuesToNotBeNull(column=col)
    )

In [10]:
add_type_and_length(expectation_suite, 'track_id', 'str', 22)

In [11]:
add_type_and_length_range(expectation_suite, 'artists', 'str', 2, 512)

In [12]:
add_type_and_length_range(expectation_suite, 'album_name', 'str', 2, 512)

In [13]:
add_type_and_length_range(expectation_suite, 'track_name', 'str', 2, 512)

In [14]:
add_type_and_range(expectation_suite, 'popularity', 'int', 0, 100)

In [15]:
add_type_and_range(expectation_suite, 'duration_ms', 'int', 0, 5237760, strict_min=True)

In [16]:
expectation_suite.add_expectation(
    gx.expectations.ExpectColumnValuesToBeOfType(column='explicit', type='bool')
);

In [17]:
add_type_and_range(expectation_suite, 'danceability', 'float', 0, 1)

In [18]:
add_type_and_range(expectation_suite, 'energy', 'float', 0, 1)

In [19]:
add_type_and_range(expectation_suite, 'key', 'int', 0, 11)

In [20]:
add_type_and_range(expectation_suite, 'loudness', 'float', -45, 5)

In [21]:
add_type_and_range(expectation_suite, 'mode', 'float', 0, 1)

In [22]:
add_type_and_range(expectation_suite, 'speechiness', 'float', 0, 1)

In [23]:
add_type_and_range(expectation_suite, 'acousticness', 'float', 0, 1)

In [24]:
add_type_and_range(expectation_suite, 'instrumentalness', 'float', 0, 1)

In [25]:
add_type_and_range(expectation_suite, 'liveness', 'float', 0, 1)

In [26]:
add_type_and_range(expectation_suite, 'valence', 'float', 0, 1)

In [27]:
add_type_and_range(expectation_suite, 'tempo', 'float', 0, 256)

In [28]:
add_type_and_range(expectation_suite, 'time_signature', 'int', 0, 5)

In [29]:
# track_genre: str, ограничение на 114 уникальных жанров
expectation_suite.add_expectation(
    gx.expectations.ExpectColumnValuesToBeOfType(column='track_genre', type='str')
)
expectation_suite.add_expectation(
    gx.expectations.ExpectColumnValuesToBeInSet(column='track_genre', value_set=list(UNIQUE_GENRES))
)
expectation_suite.add_expectation(
    gx.expectations.ExpectColumnUniqueValueCountToBeBetween(
        column='track_genre', min_value=n_genres, max_value=n_genres, strict_min=False, strict_max=False
    )
)
print(f'добавлена проверка на уникальность track_genre = {n_genres}')

добавлена проверка на уникальность track_genre = 114


In [30]:
# Сохраняем suite в JSON файл
suite_json = expectation_suite.to_json_dict()
with open('music_data_expectations.json', 'w', encoding='utf-8') as f:
    json.dump(suite_json, f, indent=2, ensure_ascii=False)
print(f'Expectation Suite сохранён в music_data_expectations.json')
print(f'Количество ожиданий: {len(expectation_suite.expectations)}')

Expectation Suite сохранён в music_data_expectations.json
Количество ожиданий: 80


In [31]:
# Запускаем валидацию через validator
validator = context.get_validator(
    batch_request=batch_request,
    expectation_suite_name=expectation_suite.name
)

validation_result = validator.validate()
print(f'Проверка завершена: {validation_result.success}')

# Сохраняем результат валидации в JSON (для Data Docs используем checkpoint ниже)
results_dict = validation_result.to_json_dict()
with open('music_data_validation_results.json', 'w', encoding='utf-8') as f:
    json.dump(results_dict, f, indent=2, ensure_ascii=False)
print('Результаты проверки сохранены в music_data_validation_results.json')

Calculating Metrics:   0%|          | 0/130 [00:00<?, ?it/s]

Проверка завершена: False
Результаты проверки сохранены в music_data_validation_results.json


In [32]:
from great_expectations.checkpoint import Checkpoint
from great_expectations.core.validation_definition import ValidationDefinition

# Получаем batch_definition из datasource
batch_definition = datasource.get_batch_definition_from_batch_request(batch_request)

# Создаем validation_definition с batch_definition вместо batch_request
validation_definition = ValidationDefinition(
    name=f"{expectation_suite.name}_validation",
    data=batch_definition,
    suite=expectation_suite
)

# Создаем checkpoint
checkpoint = Checkpoint(
    name="music_data_checkpoint",
    validation_definitions=[validation_definition]
)

# Добавляем checkpoint в контекст
context.checkpoints.add(checkpoint)

# Запускаем проверку
checkpoint_result = checkpoint.run()

# Строим HTML-отчёт
context.build_data_docs()

# Открываем в браузере
context.open_data_docs()

print("Data Docs сгенерированы!")
print(f"Путь: {context.get_docs_location()}")